In [ ]:
# ===============================
# IMPORTS
# ===============================
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (safe for all envs)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, recall_score,
                             balanced_accuracy_score, precision_score,
                             f1_score, confusion_matrix)
from sklearn.model_selection import GroupKFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from scipy.stats import skew, kurtosis, ttest_rel
from scipy.signal import welch

# ===============================
# USER SETTINGS
# ===============================
BASE_PATH             = r"C:\Users\Admin\PycharmProjects\SVM-RF-with-Sleep-Database\EEG-Seizure-Detection-With-CS"
SEGMENT_SEC           = 2
BASELINE_MINUTES      = 5
FS                    = 256
SENSITIVITY_THRESHOLD = 0.35
MIN_CHANNELS          = 15
USE_GRIDSEARCH        = True
USE_RANDOM_FOREST     = False

# Output directory for saved figures (same folder as this script by default)
OUT_DIR = r"C:\Users\Admin\PycharmProjects\SVM-RF-with-Sleep-Database\EEG-Seizure-Detection-With-CS\results"
os.makedirs(OUT_DIR, exist_ok=True)

# ===============================
# COMPRESSED SENSING SETTINGS
# ===============================
CS_RATIO       = 0.25        # Compression ratio: 0.25 → keep 25% of samples
                              # Try: 0.1, 0.25, 0.5, 0.75
CS_BASIS       = 'gaussian'  # 'gaussian' or 'bernoulli.'
CS_RANDOM_SEED = 42          # Fixed seed for reproducible Φ matrix

# ===============================
# FIXED COMMON CHANNELS  (23 total)
# ===============================
COMMON_CHANNELS = [
    'FP1-F7', 'F7-T7',  'T7-P7',    'P7-O1',
    'FP1-F3', 'F3-C3',  'C3-P3',    'P3-O1',
    'FP2-F4', 'F4-C4',  'C4-P4',    'P4-O2',
    'FP2-F8', 'F8-T8',  'T8-P8',    'P8-O2',
    'FZ-CZ',  'CZ-PZ',
    'P7-T7',  'T7-FT9', 'FT9-FT10', 'FT10-T8', 'P8-T8'
]
N_CHANNELS = len(COMMON_CHANNELS)   # 23


# =============================================================================
# COMPRESSED SENSING FUNCTIONS
# =============================================================================

def build_measurement_matrix(n_samples, cs_ratio, basis='gaussian', seed=42):
    """
    Build a CS measurement matrix Φ of shape (M, n_samples).

    M = int(n_samples × cs_ratio)  — number of compressed measurements.

    Two basic options:
      'gaussian'  : Φ_ij ~ N(0, 1/M)   — satisfies RIP with high probability
      'bernoulli' : Φ_ij ~ ±1/√M       — hardware-friendly, also RIP-compliant
    """
    M   = max(1, int(round(n_samples * cs_ratio)))
    rng = np.random.default_rng(seed)

    if basis == 'gaussian':
        Phi = rng.standard_normal((M, n_samples)) / np.sqrt(M)
    elif basis == 'bernoulli':
        Phi = (rng.integers(0, 2, size=(M, n_samples)) * 2 - 1) / np.sqrt(M)
    else:
        raise ValueError(f"Unknown CS basis '{basis}'. Use 'gaussian' or 'bernoulli'.")

    return Phi, M


def compress_segment(segment, Phi):
    """Apply CS to a single EEG segment: (n_channels, n_samples) → (n_channels, M)."""
    return segment @ Phi.T


def compress_all_segments(segments, Phi):
    """Apply CS to all segments: (n_segs, n_channels, n_samples) → (n_segs, n_channels, M)."""
    return segments @ Phi.T


# =============================================================================
# STANDARD PIPELINE FUNCTIONS
# =============================================================================

def segment_eeg(data, fs, segment_sec):
    samples = int(fs * segment_sec)
    segments, segment_times = [], []
    for start in range(0, data.shape[1] - samples + 1, samples):
        end = start + samples
        segments.append(data[:, start:end])
        segment_times.append((start / fs, end / fs))
    return np.array(segments), segment_times


def label_segments(segment_times, seizure_intervals):
    labels = []
    for seg_start, seg_end in segment_times:
        label = 0
        for onset, offset in seizure_intervals:
            if seg_start < offset and seg_end > onset:
                label = 1
                break
        labels.append(label)
    return np.array(labels)


def pad_to_fixed_channels(data, target_channels):
    n_actual, n_samples = data.shape
    if n_actual == target_channels:
        return data
    elif n_actual > target_channels:
        return data[:target_channels, :]
    else:
        pad = np.zeros((target_channels - n_actual, n_samples))
        return np.vstack([data, pad])


def extract_features(segments, fs_effective):
    """
    Extract 9 features per channel from each segment.
    Works on both raw (fs=256) and CS-compressed (fs_effective = cs_ratio × 256) segments.

    Features:
        Time-domain : L2 norm, mean, std, skewness, kurtosis
        Freq-domain : delta, theta, alpha, beta (Welch PSD)
    """
    all_features = []
    for seg in segments:
        seg_features = []
        for ch in seg:
            if np.std(ch) == 0 or not np.all(np.isfinite(ch)):
                seg_features.extend([0.0] * 9)
                continue

            l2       = np.linalg.norm(ch)
            mean     = np.mean(ch)
            std      = np.std(ch)
            sk       = skew(ch)
            kurt_val = kurtosis(ch)

            nperseg    = min(int(fs_effective), len(ch))
            freqs, psd = welch(ch, fs=fs_effective, nperseg=nperseg)

            def band_power(fmin, fmax):
                idx = (freqs >= fmin) & (freqs < fmax)
                return np.mean(psd[idx]) if idx.any() else 0.0

            delta = band_power(0.5,  4.0)
            theta = band_power(4.0,  8.0)
            alpha = band_power(8.0,  13.0)
            beta  = band_power(13.0, 30.0)

            feats = [l2, mean, std, sk, kurt_val, delta, theta, alpha, beta]
            feats = [0.0 if not np.isfinite(v) else v for v in feats]
            seg_features.extend(feats)

        all_features.append(seg_features)
    return np.array(all_features)


def parse_summary(summary_path, edf_name, patient_id):
    seizure_intervals = []
    with open(summary_path, 'r') as f:
        lines = f.readlines()
    inside_file = False
    onset = None
    for line in lines:
        if "File Name:" in line:
            current_file = line.split(":")[1].strip()
            inside_file  = (current_file == edf_name)
        if inside_file:
            if "Seizure" in line and "Start Time" in line:
                onset = int(line.split(":")[-1].strip().split()[0])
            if "Seizure" in line and "End Time" in line and onset is not None:
                offset = int(line.split(":")[-1].strip().split()[0])
                seizure_intervals.append((onset, offset))
                onset = None
    print(f"    Seizures found : {seizure_intervals}")
    return seizure_intervals


def safe_smote(X, y, random_state=42):
    classes, counts = np.unique(y, return_counts=True)
    minority_count  = counts.min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("    Falling back to random oversampling.")
        minority_class = classes[counts.argmin()]
        X_min    = X[y == minority_class]
        n_needed = counts.max() - counts.min()
        idx      = np.random.choice(len(X_min), n_needed, replace=True)
        return np.vstack([X, X_min[idx]]), np.concatenate(
            [y, np.full(n_needed, minority_class)])
    return SMOTE(random_state=random_state,
                 k_neighbors=k).fit_resample(X, y)


# =============================================================================
# VISUALISATION HELPERS
# =============================================================================

def plot_confusion_matrix(cm_total, out_dir):
    """
    Plot and save the aggregated (pooled) confusion matrix across all folds.

    Parameters
    ----------
    cm_total : np.ndarray, shape (2, 2) — summed confusion matrix
    out_dir  : str — directory to save the figure
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_total, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)

    classes   = ['Non-seizure (0)', 'Seizure (1)']
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(classes, rotation=30, ha='right')
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(classes)

    # Annotate cells
    thresh = cm_total.max() / 2.0
    for i in range(cm_total.shape[0]):
        for j in range(cm_total.shape[1]):
            ax.text(j, i, f'{cm_total[i, j]}',
                    ha='center', va='center',
                    color='white' if cm_total[i, j] > thresh else 'black',
                    fontsize=13, fontweight='bold')

    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')
    ax.set_title('Aggregated Confusion Matrix\n(pooled across all CV folds)')
    fig.tight_layout()

    path = os.path.join(out_dir, 'confusion_matrix.png')
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"\n  [Saved] Confusion matrix → {path}")


def plot_metrics_boxplot(metrics_dict, out_dir):
    """
    Boxplot showing the distribution of each metric across folds.

    Parameters
    ----------
    metrics_dict : dict  {metric_name: [fold_values]}
    out_dir      : str
    """
    fig, ax = plt.subplots(figsize=(9, 5))

    labels = list(metrics_dict.keys())
    data   = [metrics_dict[k] for k in labels]

    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))

    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    # Overlay individual fold points
    for i, d in enumerate(data):
        x = np.random.normal(i + 1, 0.04, size=len(d))
        ax.scatter(x, d, color='black', alpha=0.6, s=25, zorder=3)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_title('Per-Fold Metric Distribution (Boxplot)', fontsize=13)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    fig.tight_layout()

    path = os.path.join(out_dir, 'metrics_boxplot.png')
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [Saved] Metrics boxplot   → {path}")


def plot_sensitivity_specificity_scatter(sensitivities, specificities,
                                         balanced_accs, fold_labels, out_dir):
    """
    Scatter plot of Sensitivity vs. Specificity per fold.
    Point colour encodes balanced accuracy; ideal point is (1, 1).

    Parameters
    ----------
    sensitivities  : list of float
    specificities  : list of float
    balanced_accs  : list of float
    fold_labels    : list of str  — e.g. ['Fold 1', 'Fold 2', ...]
    out_dir        : str
    """
    fig, ax = plt.subplots(figsize=(6, 5))

    sc = ax.scatter(specificities, sensitivities,
                    c=balanced_accs, cmap='RdYlGn',
                    s=120, edgecolors='black', linewidths=0.8,
                    vmin=0, vmax=1, zorder=3)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Balanced Accuracy', fontsize=10)

    # Annotate each point with its fold label
    for i, label in enumerate(fold_labels):
        ax.annotate(label,
                    (specificities[i], sensitivities[i]),
                    textcoords='offset points', xytext=(6, 4),
                    fontsize=8, color='#333333')

    # Ideal point marker
    ax.scatter([1], [1], marker='*', s=250, color='gold',
               edgecolors='black', linewidths=0.8, zorder=4, label='Ideal (1,1)')

    # Reference lines
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)

    ax.set_xlabel('Specificity (TNR)', fontsize=12)
    ax.set_ylabel('Sensitivity (TPR)', fontsize=12)
    ax.set_xlim(-0.05, 1.10)
    ax.set_ylim(-0.05, 1.10)
    ax.set_title('Sensitivity vs. Specificity per Fold', fontsize=13)
    ax.legend(fontsize=9)
    fig.tight_layout()

    path = os.path.join(out_dir, 'sens_spec_scatter.png')
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [Saved] Scatter plot       → {path}")


def run_paired_ttest(svm_bal_accs, dummy_bal_accs):
    """
    Paired t-test: SVM balanced accuracy vs. majority-class dummy baseline.

    The majority-class dummy always predicts the most frequent class in the
    test fold.  Its balanced accuracy is 0.5 by definition when both classes
    are present (which is guaranteed by our balanced sampling step).

    Parameters
    ----------
    svm_bal_accs   : list of float — one balanced-accuracy per fold (SVM)
    dummy_bal_accs : list of float — one balanced-accuracy per fold (dummy)

    Prints a concise report and returns (t_stat, p_value).
    """
    t_stat, p_val = ttest_rel(svm_bal_accs, dummy_bal_accs)

    print("\n" + "=" * 60)
    print("  PAIRED T-TEST  —  SVM vs. Majority-Class Dummy")
    print("=" * 60)
    print(f"  SVM   balanced acc  : {np.mean(svm_bal_accs):.4f}  "
          f"±  {np.std(svm_bal_accs):.4f}")
    print(f"  Dummy balanced acc  : {np.mean(dummy_bal_accs):.4f}  "
          f"±  {np.std(dummy_bal_accs):.4f}")
    print(f"  t-statistic         : {t_stat:.4f}")
    print(f"  p-value             : {p_val:.6f}")
    if p_val < 0.05:
        print("  Result              : SIGNIFICANT (p < 0.05) — "
              "SVM outperforms dummy baseline.")
    else:
        print("  Result              : NOT significant (p ≥ 0.05).")
    print("=" * 60)
    return t_stat, p_val


# =============================================================================
# MAIN PIPELINE
# =============================================================================

# ── Step 0: Build CS measurement matrix ONCE ──────────────────────────────────
N_SAMPLES_ORIG = int(FS * SEGMENT_SEC)   # 512

Phi, M = build_measurement_matrix(
    n_samples  = N_SAMPLES_ORIG,
    cs_ratio   = CS_RATIO,
    basis      = CS_BASIS,
    seed       = CS_RANDOM_SEED
)

FS_EFFECTIVE = FS * CS_RATIO   # e.g. 256 × 0.25 = 64 Hz

print("=" * 60)
print("  COMPRESSED SENSING CONFIGURATION")
print("=" * 60)
print(f"  Original signal length : {N_SAMPLES_ORIG} samples  ({SEGMENT_SEC}s @ {FS}Hz)")
print(f"  CS ratio               : {CS_RATIO:.0%}")
print(f"  Compressed length (M)  : {M} measurements")
print(f"  Measurement matrix Φ   : shape {Phi.shape}  [{CS_BASIS}]")
print(f"  Effective fs after CS  : {FS_EFFECTIVE:.1f} Hz")
print(f"  Data reduction         : {(1 - CS_RATIO)*100:.0f}%")
print(f"  Feature vector size    : {N_CHANNELS} channels × 9 = {N_CHANNELS*9} dims")
print("=" * 60)

# ── Step 1: Load, compress, and collect all patient data ───────────────────────
all_segments    = []
all_labels      = []
all_patient_ids = []

patient_folders = sorted([f for f in os.listdir(BASE_PATH)
                           if f.lower().startswith("chb")])
print("\nAll contents of BASE_PATH :", os.listdir(BASE_PATH))
print("Patients detected          :", patient_folders)

for patient_id in patient_folders:
    print(f"\n{'='*60}")
    print(f"  PATIENT : {patient_id}")
    print(f"{'='*60}")

    patient_path  = os.path.join(BASE_PATH, patient_id)
    files         = os.listdir(patient_path)
    edf_files     = sorted([f for f in files if f.endswith(".edf")])
    summary_files = [f for f in files if "summary" in f.lower()]

    print(f"  Total files   : {len(files)}")
    print(f"  EDF files     : {len(edf_files)}")
    print(f"  Summary files : {len(summary_files)}")

    if not summary_files:
        print(f"  No summary file. Skipping {patient_id}.")
        continue

    summary_path     = os.path.join(patient_path, summary_files[0])
    patient_appended = 0

    for edf_file in edf_files:
        edf_path = os.path.join(patient_path, edf_file)
        print(f"\n  -> {edf_file}")

        try:
            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            raw.pick("eeg")
            raw.drop_channels([ch for ch in raw.ch_names if "ECG" in ch])
        except Exception as e:
            print(f"    Load failed: {e}")
            continue

        available = [ch for ch in COMMON_CHANNELS if ch in raw.ch_names]
        print(f"    Channels matched : {len(available)} / {N_CHANNELS}")

        if len(available) < MIN_CHANNELS:
            print(f"   Too few channels. Skipping.")
            continue

        raw.pick_channels(available)
        data = raw.get_data()
        fs   = int(raw.info['sfreq'])
        data = pad_to_fixed_channels(data, N_CHANNELS)

        seizure_intervals = parse_summary(summary_path, edf_file, patient_id)
        if not seizure_intervals:
            print(f"    No seizures. Skipping.")
            continue

        segments, segment_times = segment_eeg(data, fs, SEGMENT_SEC)
        labels                  = label_segments(segment_times, seizure_intervals)

        # ──  COMPRESSED SENSING  ────────────────────────────────────────────
        segments_cs = compress_all_segments(segments, Phi)

        seizure_idx = np.where(labels == 1)[0]
        if len(seizure_idx) == 0:
            print(f"    No seizure segments. Skipping.")
            continue

        X_seizure    = segments_cs[seizure_idx]
        baseline_sec = BASELINE_MINUTES * 60
        baseline_idx = []

        for onset, offset in seizure_intervals:
            baseline_start = max(0, onset - baseline_sec)
            for i, (ss, se) in enumerate(segment_times):
                if ss >= baseline_start and se <= onset and labels[i] == 0:
                    baseline_idx.append(i)

        baseline_idx = np.unique(baseline_idx)
        if len(baseline_idx) == 0:
            print(f"    No baseline segments. Skipping.")
            continue

        X_baseline = segments_cs[baseline_idx]

        print(f"    Original segment shape  : {segments.shape[1:]}")
        print(f"    CS segment shape        : {segments_cs.shape[1:]}  "
              f"[{N_SAMPLES_ORIG}->{M} per channel]")
        print(f"    Seizure  segments       : {len(X_seizure)}")
        print(f"    Baseline segments       : {len(X_baseline)}")

        n_final = min(len(X_seizure), len(X_baseline))
        sel     = np.random.choice(len(X_baseline), n_final, replace=False)
        X_bal   = np.concatenate([X_seizure[:n_final], X_baseline[sel]])
        y_bal   = np.concatenate([np.ones(n_final), np.zeros(n_final)])

        all_segments.append(X_bal)
        all_labels.append(y_bal)
        all_patient_ids.append(np.full(len(y_bal), patient_id))
        patient_appended += 1

    print(f"\n {patient_id} : {patient_appended} EDF file(s) contributed data.")

if not all_segments:
    raise RuntimeError(
        "No data collected. Check folder structure, "
        "channel names, and summary files."
    )

# ── Step 2: Concatenate ────────────────────────────────────────────────────────
X_all  = np.concatenate(all_segments)
y_all  = np.concatenate(all_labels)
groups = np.concatenate(all_patient_ids)

print(f"\nConcatenated CS segment shape : {X_all.shape}")
print(f"Class balance                  : {np.unique(y_all, return_counts=True)}")

# ── Step 3: Feature extraction ────────────────────────────────────────────────
print("\nExtracting features from CS-compressed segments...")
X_features = extract_features(X_all, fs_effective=FS_EFFECTIVE)
print(f"Feature matrix shape : {X_features.shape}")

# ── Step 4: NaN / Inf cleanup ─────────────────────────────────────────────────
print("\nCleaning NaN / Inf values...")
finite_mask = np.isfinite(X_features).all(axis=1)
n_dropped   = (~finite_mask).sum()
if n_dropped > 0:
    print(f"  Dropping {n_dropped} non-finite samples.")
    X_features = X_features[finite_mask]
    y_all      = y_all[finite_mask]
    groups     = groups[finite_mask]

print(f"  Any NaNs remaining : {np.isnan(X_features).any()}")
print(f"  Clean shape        : {X_features.shape}")
print(f"  Class balance      : {np.unique(y_all, return_counts=True)}")

# ── Step 5: GroupKFold setup ───────────────────────────────────────────────────
n_real    = len(np.unique(groups))
X_cv      = X_features
y_cv      = y_all
groups_cv = groups

if n_real < 2:
    print("Only one patient. Using StratifiedKFold(5).")
    cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits      = list(cv_splitter.split(X_cv, y_cv))
else:
    print(f"\nUsing GroupKFold with {n_real} real patients.")
    gkf    = GroupKFold(n_splits=n_real)
    splits = list(gkf.split(X_cv, y_cv, groups_cv))

# ── Step 6: Cross-validation loop ─────────────────────────────────────────────
accuracies     = []
sensitivities  = []
specificities  = []
balanced_accs  = []
precisions     = []
f1_scores      = []
dummy_bal_accs = []          # majority-class baseline per fold
fold_labels    = []          # e.g. "Fold 1 (chb01)"
cm_total       = np.zeros((2, 2), dtype=int)   # aggregated confusion matrix

print("\nRunning cross-validation folds...\n")

for fold_i, (train_idx, test_idx) in enumerate(splits):
    X_train, X_test = X_cv[train_idx], X_cv[test_idx]
    y_train, y_test = y_cv[train_idx], y_cv[test_idx]
    test_patient    = np.unique(groups_cv[test_idx])

    fold_label = f"Fold {fold_i+1}\n({', '.join(test_patient)})"
    fold_labels.append(fold_label)

    # ── Per-fold StandardScaler ───────────────────────────────────────────────
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ── Per-fold SMOTE ────────────────────────────────────────────────────────
    print(f"  Fold {fold_i+1} | Test patient: {test_patient}")
    X_train, y_train = safe_smote(X_train, y_train)
    print(f"    After SMOTE — train shape: {X_train.shape} | "
          f"balance: {np.unique(y_train, return_counts=True)}")

    # ── Classifier ───────────────────────────────────────────────────────────
    if USE_RANDOM_FOREST:
        clf = RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        clf.fit(X_train, y_train)
        y_proba = clf.predict_proba(X_test)[:, 1]

    elif USE_GRIDSEARCH:
        param_grid = {
            'C'    : [1, 10, 100],
            'gamma': ['scale', 0.01, 0.001]
        }
        base_clf = SVC(kernel='rbf', class_weight='balanced', probability=True)
        grid     = GridSearchCV(
            base_clf, param_grid,
            cv=3, scoring='balanced_accuracy', n_jobs=-1
        )
        grid.fit(X_train, y_train)
        clf     = grid.best_estimator_
        y_proba = clf.predict_proba(X_test)[:, 1]
        print(f"    Best params : {grid.best_params_}")

    else:
        clf = SVC(kernel='rbf', C=10, gamma='scale',
                  class_weight='balanced', probability=True)
        clf.fit(X_train, y_train)
        y_proba = clf.predict_proba(X_test)[:, 1]

    # ── Threshold-based predictions ───────────────────────────────────────────
    y_pred = (y_proba >= SENSITIVITY_THRESHOLD).astype(int)

    acc     = accuracy_score(y_test, y_pred)
    sens    = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    spec    = recall_score(y_test, y_pred, pos_label=0, zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1      = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

    accuracies.append(acc)
    sensitivities.append(sens)
    specificities.append(spec)
    balanced_accs.append(bal_acc)
    precisions.append(prec)
    f1_scores.append(f1)

    # ── Confusion matrix accumulation ─────────────────────────────────────────
    cm_fold  = confusion_matrix(y_test, y_pred, labels=[0, 1])
    cm_total += cm_fold

    # ── Dummy baseline (majority-class) ───────────────────────────────────────
    # Predict majority class of test fold for every test sample.
    # balanced_accuracy of a majority-class predictor = 0.5 when both
    # classes are present (which our balanced pipeline ensures).
    majority_class = int(np.bincount(y_test.astype(int)).argmax())
    y_dummy        = np.full_like(y_test, majority_class)
    dummy_bal      = balanced_accuracy_score(y_test, y_dummy)
    dummy_bal_accs.append(dummy_bal)

    print(f"    Acc: {acc:.4f} | Sens: {sens:.4f} | Spec: {spec:.4f} | "
          f"Prec: {prec:.4f} | F1: {f1:.4f} | BalAcc: {bal_acc:.4f}  "
          f"[Dummy BalAcc: {dummy_bal:.4f}]\n")

# ── Step 7: Final numeric results ─────────────────────────────────────────────
print("=" * 60)
print("  FINAL RESULTS — WITH COMPRESSED SENSING  (per-fold SMOTE only)")
print("=" * 60)
print(f"  CS ratio           : {CS_RATIO:.0%}  "
      f"({N_SAMPLES_ORIG} → {M} samples per channel)")
print(f"  CS basis           : {CS_BASIS}")
print(f"  Feature dims       : {N_CHANNELS * 9}  (same structure, CS-compressed input)")
print(f"  Data reduction     : {(1 - CS_RATIO)*100:.0f}% of original signal")
print("-" * 60)
print(f"  Accuracy           : {np.mean(accuracies):.4f}  ±  {np.std(accuracies):.4f}")
print(f"  Sensitivity        : {np.mean(sensitivities):.4f}  ±  {np.std(sensitivities):.4f}")
print(f"  Specificity        : {np.mean(specificities):.4f}  ±  {np.std(specificities):.4f}")
print(f"  Balanced Accuracy  : {np.mean(balanced_accs):.4f}  ±  {np.std(balanced_accs):.4f}")
print(f"  Precision          : {np.mean(precisions):.4f}  ±  {np.std(precisions):.4f}")
print(f"  F1 Score           : {np.mean(f1_scores):.4f}  ±  {np.std(f1_scores):.4f}")
print("=" * 60)

# ── Step 8: Confusion matrix ──────────────────────────────────────────────────
print("\n  Aggregated Confusion Matrix (pooled across all folds):")
print(f"  [[TN={cm_total[0,0]}  FP={cm_total[0,1]}]")
print(f"   [FN={cm_total[1,0]}  TP={cm_total[1,1]}]]")
plot_confusion_matrix(cm_total, OUT_DIR)

# ── Step 9: Boxplot ───────────────────────────────────────────────────────────
metrics_dict = {
    'Accuracy'   : accuracies,
    'Sensitivity': sensitivities,
    'Specificity': specificities,
    'Bal. Acc.'  : balanced_accs,
    'Precision'  : precisions,
    'F1 Score'   : f1_scores,
}
plot_metrics_boxplot(metrics_dict, OUT_DIR)

# ── Step 10: Sensitivity vs. Specificity scatter ──────────────────────────────
plot_sensitivity_specificity_scatter(
    sensitivities, specificities, balanced_accs, fold_labels, OUT_DIR
)

# ── Step 11: Paired t-test ────────────────────────────────────────────────────
run_paired_ttest(balanced_accs, dummy_bal_accs)

print(f"\nAll figures saved to: {OUT_DIR}")